In [65]:
%pip install pyspark==4.0.1 findspark

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [66]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import split, col
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("big-data-programming-2.3.2")
    .master("local[*]")
    .getOrCreate()
)

In [67]:
df = spark.read.csv(
    "./input/weatherData.csv",
    header=True,
    inferSchema=True,
)

df.show()

+-----------+---------+-----------------------+-----------------------+-----------------------+------------------------+-----------------------------+-----------------------------+------------------------------+---------------------+---------------------+----------------------+-------------+-----------------------+-------------------------+-------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------+-------------------+
|location_id|     date|weather_code (wmo code)|temperature_2m_max (°C)|temperature_2m_min (°C)|temperature_2m_mean (°C)|apparent_temperature_max (°C)|apparent_temperature_min (°C)|apparent_temperature_mean (°C)|daylight_duration (s)|sunshine_duration (s)|precipitation_sum (mm)|rain_sum (mm)|precipitation_hours (h)|wind_speed_10m_max (km/h)|wind_gusts_10m_max (km/h)|wind_direction_10m_dominant (°)|shortwave_radiation_sum (MJ/m²)|et0_fao_evapotranspiration (mm)|            sunrise|           

In [68]:
df = df.withColumn("year", split(col("date"), "/").getItem(2))
df = df.withColumn("month", split(col("date"), "/").getItem(0))
df = df.withColumn("day", split(col("date"), "/").getItem(1))
df = df.select("year", "month", "day", "temperature_2m_max (°C)")
df.show()

+----+-----+---+-----------------------+
|year|month|day|temperature_2m_max (°C)|
+----+-----+---+-----------------------+
|2010|    1|  1|                   30.1|
|2010|    1|  2|                   30.1|
|2010|    1|  3|                   29.6|
|2010|    1|  4|                   28.9|
|2010|    1|  5|                   28.1|
|2010|    1|  6|                   28.8|
|2010|    1|  7|                   29.3|
|2010|    1|  8|                   28.4|
|2010|    1|  9|                   29.1|
|2010|    1| 10|                   30.1|
|2010|    1| 11|                   29.7|
|2010|    1| 12|                   29.8|
|2010|    1| 13|                   30.0|
|2010|    1| 14|                   30.1|
|2010|    1| 15|                   30.7|
|2010|    1| 16|                   31.1|
|2010|    1| 17|                   29.8|
|2010|    1| 18|                   29.5|
|2010|    1| 19|                   31.0|
|2010|    1| 20|                   31.4|
+----+-----+---+-----------------------+
only showing top

In [69]:
df = df.withColumn("week", ((col("day").cast("int") - 1) / 7).cast("int") + 1)
df.show()

+----+-----+---+-----------------------+----+
|year|month|day|temperature_2m_max (°C)|week|
+----+-----+---+-----------------------+----+
|2010|    1|  1|                   30.1|   1|
|2010|    1|  2|                   30.1|   1|
|2010|    1|  3|                   29.6|   1|
|2010|    1|  4|                   28.9|   1|
|2010|    1|  5|                   28.1|   1|
|2010|    1|  6|                   28.8|   1|
|2010|    1|  7|                   29.3|   1|
|2010|    1|  8|                   28.4|   2|
|2010|    1|  9|                   29.1|   2|
|2010|    1| 10|                   30.1|   2|
|2010|    1| 11|                   29.7|   2|
|2010|    1| 12|                   29.8|   2|
|2010|    1| 13|                   30.0|   2|
|2010|    1| 14|                   30.1|   2|
|2010|    1| 15|                   30.7|   3|
|2010|    1| 16|                   31.1|   3|
|2010|    1| 17|                   29.8|   3|
|2010|    1| 18|                   29.5|   3|
|2010|    1| 19|                  

In [70]:
average_max_temperatures = (
    df.groupBy("year", "month")
    .avg("temperature_2m_max (°C)")
    .withColumnRenamed("avg(temperature_2m_max (°C))", "avg_temp")
)
average_max_temperatures.show()

+----+-----+------------------+
|year|month|          avg_temp|
+----+-----+------------------+
|2014|    2|28.869047619047628|
|2012|   11|27.777777777777775|
|2014|   11|27.189629629629618|
|2010|    1| 27.48912783751493|
|2014|    8|29.556869772998795|
|2010|    3| 31.24097968936678|
|2017|    7| 31.07801672640383|
|2012|    7|30.324253285543602|
|2020|    5|30.507526881720423|
|2020|   11|28.393827160493817|
|2022|   11|27.852345679012352|
|2010|    4|30.652222222222218|
|2014|    5| 29.81433691756273|
|2020|   10| 30.24516129032258|
|2011|    4|29.432222222222226|
|2016|    7|29.932616487455196|
|2012|    8| 30.23906810035843|
|2014|   12| 26.34671445639188|
|2015|   11|27.610987654320986|
|2023|   11|28.252962962962965|
+----+-----+------------------+
only showing top 20 rows


In [71]:
window = Window.partitionBy("year")

hottest_months = (
    average_max_temperatures
    .withColumn("max_avg_temp", F.max("avg_temp").over(window))
    .filter(F.col("avg_temp") == F.col("max_avg_temp"))
    .drop("max_avg_temp")
    .withColumnRenamed("avg_temp", "hottest_month_avg_temp")
    .withColumnRenamed("month", "hottest_month")
)

hottest_months.show()

+----+-------------+----------------------+
|year|hottest_month|hottest_month_avg_temp|
+----+-------------+----------------------+
|2010|            3|     31.24097968936678|
|2011|            6|    30.435432098765425|
|2012|            5|     31.04026284348865|
|2013|            4|    30.644814814814815|
|2014|            4|    30.782222222222217|
|2015|            7|    30.113261648745524|
|2016|            4|    31.960617283950615|
|2017|            4|     31.72814814814815|
|2018|            4|    30.452962962962964|
|2019|            5|    31.549940262843492|
|2020|            3|     31.44360812425328|
|2021|            4|    30.458395061728403|
|2022|            3|     30.01911589008363|
|2023|            8|     31.86523297491039|
|2024|            4|    32.497407407407394|
+----+-------------+----------------------+



In [72]:
joined_df = hottest_months.join(
    df,
    (hottest_months.year == df.year) & (hottest_months.hottest_month == df.month),
    how="inner"
)
joined_df = joined_df.select(hottest_months["year"], "hottest_month", "day", "week", "temperature_2m_max (°C)")
joined_df.show()

+----+-------------+---+----+-----------------------+
|year|hottest_month|day|week|temperature_2m_max (°C)|
+----+-------------+---+----+-----------------------+
|2010|            3|  1|   1|                   32.6|
|2010|            3|  2|   1|                   32.1|
|2010|            3|  3|   1|                   32.7|
|2010|            3|  4|   1|                   30.4|
|2010|            3|  5|   1|                   31.6|
|2010|            3|  6|   1|                   31.6|
|2010|            3|  7|   1|                   32.9|
|2010|            3|  8|   2|                   32.7|
|2010|            3|  9|   2|                   33.6|
|2010|            3| 10|   2|                   33.5|
|2010|            3| 11|   2|                   34.9|
|2010|            3| 12|   2|                   34.0|
|2010|            3| 13|   2|                   34.0|
|2010|            3| 14|   2|                   32.1|
|2010|            3| 15|   3|                   31.7|
|2010|            3| 16|   3

In [73]:
hottest_weekly_temps_df = (
    joined_df
    .groupBy("year", "hottest_month", "week")
    .avg("temperature_2m_max (°C)")
    .withColumnRenamed("avg(temperature_2m_max (°C))", "avg_weekly_temp")
    .orderBy("year", "week")
)
hottest_weekly_temps_df.show()

+----+-------------+----+------------------+
|year|hottest_month|week|   avg_weekly_temp|
+----+-------------+----+------------------+
|2010|            3|   1| 31.04021164021165|
|2010|            3|   2|31.585185185185185|
|2010|            3|   3|31.695238095238093|
|2010|            3|   4| 30.95978835978836|
|2010|            3|   5| 30.50246913580247|
|2011|            6|   1| 29.97089947089947|
|2011|            6|   2| 29.96455026455027|
|2011|            6|   3|30.729629629629628|
|2011|            6|   4| 30.83227513227513|
|2011|            6|   5| 31.29074074074074|
|2012|            5|   1|30.705820105820102|
|2012|            5|   2| 30.99470899470899|
|2012|            5|   3| 31.69259259259259|
|2012|            5|   4| 30.67883597883598|
|2012|            5|   5|31.248148148148154|
|2013|            4|   1|30.643915343915342|
|2013|            4|   2| 30.48465608465608|
|2013|            4|   3|31.022222222222226|
|2013|            4|   4|  30.4031746031746|
|2013|    

In [74]:
# transpose by week
final_df = hottest_weekly_temps_df.groupBy("year", "hottest_month").pivot("week").agg(F.max("avg_weekly_temp"))
final_df = final_df.orderBy("year", "hottest_month")
final_df = (final_df
            .withColumnRenamed("1", "week_1_max_temp")
            .withColumnRenamed("2", "week_2_max_temp")
            .withColumnRenamed("3", "week_3_max_temp")
            .withColumnRenamed("4", "week_4_max_temp")
            .withColumnRenamed("5", "week_5_max_temp")
            )
final_df.show()

+----+-------------+------------------+------------------+------------------+------------------+------------------+
|year|hottest_month|   week_1_max_temp|   week_2_max_temp|   week_3_max_temp|   week_4_max_temp|   week_5_max_temp|
+----+-------------+------------------+------------------+------------------+------------------+------------------+
|2010|            3| 31.04021164021165|31.585185185185185|31.695238095238093| 30.95978835978836| 30.50246913580247|
|2011|            6| 29.97089947089947| 29.96455026455027|30.729629629629628| 30.83227513227513| 31.29074074074074|
|2012|            5|30.705820105820102| 30.99470899470899| 31.69259259259259| 30.67883597883598|31.248148148148154|
|2013|            4|30.643915343915342| 30.48465608465608|31.022222222222226|  30.4031746031746| 30.73333333333333|
|2014|            4| 31.25714285714286|30.774603174603172|30.949735449735453|30.586243386243392|29.246296296296297|
|2015|            7| 30.45396825396825|30.438095238095233|29.67830687830

In [75]:
# spark.stop()